# JR_IBEX_003 Spatial Processing Pipeline

**Core spatial immunofluorescence analysis with advanced methods**

This workflow provides comprehensive spatial analysis using:
- Integration with existing IBEX alignment pipeline
- Advanced spatial neighborhood analysis
- Multi-scale spatial feature extraction
- Literature-based analysis methods

---

## Prerequisites
- Run `00_daily_startup.ipynb` first
- Have aligned IBEX data available
- Set current sample using `set_current_sample(roi_id)`

---

In [ ]:
# Cell 1: Initialize Spatial Processing Pipeline
print("🔬 JR_IBEX_003 Spatial Processing Pipeline")
print("=" * 50)

# Check if daily startup was run
try:
    # These should be available from daily startup
    assert 'config' in globals(), "Run 00_daily_startup.ipynb first"
    assert 'literature_methods' in globals(), "Literature methods not loaded"
    print("✅ Daily startup detected")
except (AssertionError, NameError) as e:
    print(f"⚠ {e}")
    print("Loading essential components...")
    
    # Minimal setup if daily startup not run
    import sys
    from pathlib import Path
    project_root = Path().absolute()
    sys.path.insert(0, str(project_root / "scripts"))
    
    from load_workspace import main as load_workspace
    workspace_data = load_workspace()

# Import essential libraries for spatial processing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Update progress
try:
    from scripts.track_progress import update_progress
    update_progress('01_spatial_processing', 'running', 10)
except:
    pass

In [ ]:
# Cell 2: Load and Validate Sample Data
print("📊 Loading sample data for spatial processing...")

# Get current sample
try:
    from scripts.resume_session import load_daily_state
    state = load_daily_state()
    current_sample = state.get('current_sample', {}).get('roi_id')
except:
    current_sample = None

if not current_sample:
    print("⚠ No current sample set. Available options:")
    
    # Look for available samples
    aligned_dir = Path("aligned")
    if aligned_dir.exists():
        sample_dirs = [d.name for d in aligned_dir.iterdir() if d.is_dir()]
        if sample_dirs:
            print(f"   Available samples: {sample_dirs}")
            print(f"   Set sample: set_current_sample('{sample_dirs[0]}')")
            # Auto-select first sample for demo
            current_sample = sample_dirs[0]
            print(f"   🎯 Auto-selected: {current_sample}")
        else:
            print("   No aligned samples found")
    else:
        print("   No aligned directory found")

if current_sample:
    print(f"\n📊 Processing sample: {current_sample}")
    
    # Look for aligned data
    sample_path = Path("aligned") / current_sample
    if sample_path.exists():
        aligned_files = list(sample_path.glob("*.tif"))
        composite_files = list(sample_path.glob("*ch_*.tif"))
        
        print(f"   📁 Sample path: {sample_path}")
        print(f"   📄 Aligned files: {len(aligned_files)}")
        if composite_files:
            print(f"   🔗 Composite files: {[f.name for f in composite_files]}")
        
        # Store sample info
        sample_info = {
            'roi_id': current_sample,
            'path': sample_path,
            'aligned_files': aligned_files,
            'composite_files': composite_files
        }
    else:
        print(f"   ❌ Sample path not found: {sample_path}")
        sample_info = None
else:
    print("❌ No sample available for processing")
    sample_info = None

# Update progress
try:
    update_progress('01_spatial_processing', 'running', 20)
except:
    pass

In [ ]:
# Cell 3: Integration with Existing IBEX Pipeline
print("🔧 Integrating with existing IBEX processing pipeline...")

if sample_info:
    # Check for existing IBEX processing results
    
    # Look for segmentation results
    segmentation_files = list(sample_info['path'].glob("*segmentation*"))
    if segmentation_files:
        print(f"   ✅ Found segmentation: {[f.name for f in segmentation_files]}")
    else:
        print("   ⚠ No segmentation files found")
        print("   💡 Consider running cell segmentation first")
    
    # Look for cell measurements
    measurement_files = list(sample_info['path'].glob("*measurements*")) + \
                       list(sample_info['path'].glob("*celldata*")) + \
                       list(sample_info['path'].glob("*.csv"))
    
    if measurement_files:
        print(f"   ✅ Found measurements: {[f.name for f in measurement_files]}")
        
        # Try to load cell data
        cell_data = None
        for mfile in measurement_files:
            try:
                if mfile.suffix == '.csv':
                    cell_data = pd.read_csv(mfile)
                    print(f"   📊 Loaded cell data: {len(cell_data)} cells from {mfile.name}")
                    print(f"   📊 Columns: {list(cell_data.columns[:10])}{'...' if len(cell_data.columns) > 10 else ''}")
                    break
            except Exception as e:
                print(f"   ⚠ Failed to load {mfile.name}: {e}")
    else:
        print("   ⚠ No cell measurement files found")
        print("   💡 You may need to run cell detection/measurement first")
        
        # Create synthetic data for demonstration
        print("   🎯 Creating synthetic cell data for method demonstration...")
        np.random.seed(42)
        n_cells = 1000
        
        cell_data = pd.DataFrame({
            'CellID': range(1, n_cells + 1),
            'X': np.random.randn(n_cells) * 200 + 500,
            'Y': np.random.randn(n_cells) * 200 + 500,
            'Area': np.random.exponential(50, n_cells) + 20,
            'DAPI': np.random.exponential(1000, n_cells) + 500,
            'CD45': np.random.exponential(200, n_cells),
            'CD3': np.random.exponential(150, n_cells),
            'CD68': np.random.exponential(100, n_cells),
            'IBA1': np.random.exponential(180, n_cells),
            'GFAP': np.random.exponential(120, n_cells),
            'NeuN': np.random.exponential(300, n_cells)
        })
        
        # Add some spatial structure
        cluster_centers = [(400, 400), (600, 600), (500, 300)]
        for i, (cx, cy) in enumerate(cluster_centers):
            cluster_mask = np.random.rand(n_cells) < 0.15
            cell_data.loc[cluster_mask, 'X'] = np.random.randn(np.sum(cluster_mask)) * 50 + cx
            cell_data.loc[cluster_mask, 'Y'] = np.random.randn(np.sum(cluster_mask)) * 50 + cy
        
        print(f"   📊 Created synthetic data: {len(cell_data)} cells")

else:
    print("❌ No sample info available - skipping IBEX integration")
    cell_data = None

# Update progress
try:
    update_progress('01_spatial_processing', 'running', 30)
except:
    pass

In [ ]:
# Cell 4: Basic Spatial Analysis
print("📊 Running basic spatial analysis...")

if cell_data is not None:
    # Import spatial analysis tools
    try:
        from scripts.spatial_utils import SpatialAnalyzer, quick_spatial_analysis
        
        # Run quick spatial analysis
        spatial_results = quick_spatial_analysis(
            cell_data, 
            coord_columns=['X', 'Y'],
            cell_type_column=None,  # Will add cell typing later
            pixel_size_um=0.325
        )
        
        print("\n📊 Basic Spatial Analysis Results:")
        print(f"   • Neighborhoods analyzed: {len(spatial_results['neighborhoods'])}")
        print(f"   • Mean neighborhood density: {spatial_results['neighborhoods']['neighborhood_density'].mean():.4f} cells/μm²")
        print(f"   • Mean nearest neighbor distance: {spatial_results['nearest_neighbors']['mean_distance']:.1f} μm")
        print(f"   • Spatial clusters found: {len(set(spatial_results['clustering']['cluster_id'].unique())) - 1}")
        
        # Quick visualization
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Plot 1: Cell distribution
        coords = cell_data[['X', 'Y']].values * 0.325  # Convert to microns
        axes[0].scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.7)
        axes[0].set_xlabel('X (μm)')
        axes[0].set_ylabel('Y (μm)')
        axes[0].set_title(f'Cell Distribution ({len(cell_data)} cells)')
        axes[0].axis('equal')
        
        # Plot 2: Neighborhood density
        density = spatial_results['neighborhoods']['neighborhood_density']
        axes[1].hist(density, bins=30, alpha=0.7, color='blue')
        axes[1].axvline(density.mean(), color='red', linestyle='--', 
                       label=f'Mean: {density.mean():.4f}')
        axes[1].set_xlabel('Neighborhood Density (cells/μm²)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Neighborhood Density Distribution')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()
        
    except ImportError as e:
        print(f"⚠ Spatial analysis tools not available: {e}")
        spatial_results = None
        
else:
    print("❌ No cell data available for spatial analysis")
    spatial_results = None

# Update progress
try:
    update_progress('01_spatial_processing', 'running', 50)
except:
    pass

In [ ]:
# Cell 5: Multi-scale Spatial Feature Extraction
print("🔍 Extracting multi-scale spatial features...")

if cell_data is not None:
    try:
        # Multi-scale analysis at different radii
        scale_levels = [10, 25, 50, 100, 200]  # microns
        multi_scale_results = {}
        
        analyzer = SpatialAnalyzer(pixel_size_um=0.325)
        analyzer.load_cell_data(cell_data, coord_columns=['X', 'Y'])
        
        for scale in scale_levels:
            print(f"   🔬 Analyzing scale: {scale} μm")
            
            # Neighborhood analysis at this scale
            neighborhoods = analyzer.spatial_neighborhoods(radius_um=scale)
            
            # Ripley's K at this scale
            ripley = analyzer.ripley_k_function(radii_um=[scale])
            
            multi_scale_results[scale] = {
                'mean_neighbors': neighborhoods['n_neighbors'].mean(),
                'mean_density': neighborhoods['neighborhood_density'].mean(),
                'std_density': neighborhoods['neighborhood_density'].std(),
                'ripley_l_deviation': ripley['L_deviation'].iloc[0] if not ripley.empty else 0
            }
        
        # Create multi-scale feature summary
        print("\n📊 Multi-scale Feature Summary:")
        for scale, features in multi_scale_results.items():
            print(f"   {scale:3d} μm: density={features['mean_density']:.4f}, L-dev={features['ripley_l_deviation']:+.1f}")
        
        # Visualize multi-scale features
        scales = list(multi_scale_results.keys())
        densities = [multi_scale_results[s]['mean_density'] for s in scales]
        l_deviations = [multi_scale_results[s]['ripley_l_deviation'] for s in scales]
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Plot density vs scale
        axes[0].plot(scales, densities, 'o-', color='blue')
        axes[0].set_xlabel('Scale (μm)')
        axes[0].set_ylabel('Mean Density (cells/μm²)')
        axes[0].set_title('Density vs Spatial Scale')
        axes[0].grid(True, alpha=0.3)
        
        # Plot L-function deviation vs scale
        axes[1].plot(scales, l_deviations, 'o-', color='red')
        axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
        axes[1].set_xlabel('Scale (μm)')
        axes[1].set_ylabel('L(r) - r')
        axes[1].set_title('Spatial Clustering vs Scale')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"⚠ Multi-scale analysis error: {e}")
        multi_scale_results = None
else:
    print("❌ No cell data for multi-scale analysis")
    multi_scale_results = None

# Update progress
try:
    update_progress('01_spatial_processing', 'running', 70)
except:
    pass

In [ ]:
# Cell 6: Cell Type Classification for Advanced Analysis
print("🏷️ Performing cell type classification...")

if cell_data is not None:
    # Simple marker-based cell type classification
    cell_data_classified = cell_data.copy()
    
    # Define thresholds based on data distribution
    threshold_percentile = 75
    
    # Initialize cell types
    cell_data_classified['cell_type'] = 'other'
    
    # Immune cells (CD45+)
    if 'CD45' in cell_data.columns:
        cd45_threshold = np.percentile(cell_data['CD45'], threshold_percentile)
        immune_mask = cell_data['CD45'] > cd45_threshold
        cell_data_classified.loc[immune_mask, 'cell_type'] = 'immune'
    
    # T cells (CD3+)
    if 'CD3' in cell_data.columns:
        cd3_threshold = np.percentile(cell_data['CD3'], threshold_percentile)
        t_cell_mask = cell_data['CD3'] > cd3_threshold
        cell_data_classified.loc[t_cell_mask, 'cell_type'] = 'T_cell'
    
    # Macrophages (CD68+)
    if 'CD68' in cell_data.columns:
        cd68_threshold = np.percentile(cell_data['CD68'], threshold_percentile)
        macro_mask = cell_data['CD68'] > cd68_threshold
        cell_data_classified.loc[macro_mask, 'cell_type'] = 'macrophage'
    
    # Microglia (IBA1+)
    if 'IBA1' in cell_data.columns:
        iba1_threshold = np.percentile(cell_data['IBA1'], threshold_percentile)
        microglia_mask = cell_data['IBA1'] > iba1_threshold
        cell_data_classified.loc[microglia_mask, 'cell_type'] = 'microglia'
    
    # Neurons (NeuN+, CD45-)
    if 'NeuN' in cell_data.columns:
        neun_threshold = np.percentile(cell_data['NeuN'], threshold_percentile)
        cd45_low = cell_data.get('CD45', 0) < np.percentile(cell_data.get('CD45', [0]), 25)
        neuron_mask = (cell_data['NeuN'] > neun_threshold) & cd45_low
        cell_data_classified.loc[neuron_mask, 'cell_type'] = 'neuron'
    
    # Astrocytes (GFAP+, CD45-)
    if 'GFAP' in cell_data.columns:
        gfap_threshold = np.percentile(cell_data['GFAP'], threshold_percentile)
        cd45_low = cell_data.get('CD45', 0) < np.percentile(cell_data.get('CD45', [0]), 25)
        astrocyte_mask = (cell_data['GFAP'] > gfap_threshold) & cd45_low
        cell_data_classified.loc[astrocyte_mask, 'cell_type'] = 'astrocyte'
    
    # Show classification results
    type_counts = cell_data_classified['cell_type'].value_counts()
    print("\n🏷️ Cell Type Classification Results:")
    for cell_type, count in type_counts.items():
        percentage = count / len(cell_data_classified) * 100
        print(f"   {cell_type}: {count} cells ({percentage:.1f}%)")
    
    # Visualize cell types
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Spatial distribution by cell type
    coords = cell_data_classified[['X', 'Y']].values * 0.325
    cell_types = cell_data_classified['cell_type']
    
    unique_types = cell_types.unique()
    colors = plt.cm.Set1(np.linspace(0, 1, len(unique_types)))
    
    for i, cell_type in enumerate(unique_types):
        mask = cell_types == cell_type
        axes[0].scatter(coords[mask, 0], coords[mask, 1], 
                       c=[colors[i]], label=cell_type, s=2, alpha=0.7)
    
    axes[0].set_xlabel('X (μm)')
    axes[0].set_ylabel('Y (μm)')
    axes[0].set_title('Spatial Distribution by Cell Type')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0].axis('equal')
    
    # Cell type composition
    axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%')
    axes[1].set_title('Cell Type Composition')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ No cell data for classification")
    cell_data_classified = None

# Update progress
try:
    update_progress('01_spatial_processing', 'running', 85)
except:
    pass

In [ ]:
# Cell 7: Save Results and Prepare for Advanced Analysis
print("💾 Saving spatial processing results...")

if sample_info and cell_data is not None:
    # Create results directory
    results_dir = Path("results") / "spatial_analysis" / sample_info['roi_id']
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # Save processed cell data
    if cell_data_classified is not None:
        output_path = results_dir / "spatial_processed_cells.csv"
        cell_data_classified.to_csv(output_path, index=False)
        print(f"✅ Saved processed cell data: {output_path}")
    
    # Save spatial analysis results
    if spatial_results:
        # Save neighborhoods
        neighborhoods_path = results_dir / "spatial_neighborhoods.csv"
        spatial_results['neighborhoods'].to_csv(neighborhoods_path, index=False)
        
        # Save clustering results
        clustering_path = results_dir / "spatial_clustering.csv"
        spatial_results['clustering'].to_csv(clustering_path, index=False)
        
        print(f"✅ Saved spatial analysis: neighborhoods, clustering")
    
    # Save multi-scale results
    if multi_scale_results:
        multiscale_path = results_dir / "multiscale_features.json"
        import json
        with open(multiscale_path, 'w') as f:
            json.dump(multi_scale_results, f, indent=2)
        print(f"✅ Saved multi-scale features: {multiscale_path}")
    
    print(f"\n📁 All results saved to: {results_dir}")

# Create checkpoint
try:
    from scripts.track_progress import checkpoint
    checkpoint_path = checkpoint(
        "spatial_processing_complete", 
        f"Completed spatial processing for {current_sample if current_sample else 'sample'}"
    )
    print(f"📌 Checkpoint created: spatial_processing_complete")
except:
    pass

# Prepare summary for next workflows
processing_summary = {
    'sample_id': current_sample,
    'n_cells': len(cell_data) if cell_data is not None else 0,
    'cell_types_identified': len(cell_data_classified['cell_type'].unique()) if cell_data_classified is not None else 0,
    'spatial_analysis_complete': spatial_results is not None,
    'multiscale_analysis_complete': multi_scale_results is not None,
    'ready_for_advanced_methods': True
}

print("\n📊 Processing Summary:")
for key, value in processing_summary.items():
    print(f"   {key}: {value}")

print("\n🚀 Next Steps:")
print("   1. Run 02_kipnis_analysis.ipynb for brain-immune interaction analysis")
print("   2. Run 03_germain_analysis.ipynb for multi-scale network analysis")
print("   3. Continue with comparative analysis or publication figures")

# Complete workflow
try:
    update_progress('01_spatial_processing', 'completed', 100)
    print("\n✅ Spatial processing pipeline completed successfully!")
except:
    print("\n✅ Spatial processing pipeline completed!")